# SI4006 · Entrega M3 — Un sistema RAG que recupera, actúa y se evalúa
### Equipo Offside — Jean Carlo Londoño Ocampo · Alejandro Garcés Ramírez · Nicolás Ospina Torres

**En una frase:** Offside responde las preguntas de un apostador recreativo sobre noticias de fútbol
(«¿juega Vinicius el Clásico?») buscando en la prensa recolectada, y una respuesta es buena cuando
**dice lo que dicen las noticias, cita de dónde lo saca, no se inventa nada y, si no lo sabe, lo
dice**.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JeanCarloLond/offside-londono-garces-ospina/blob/main/proyecto1/notebooks/M3_RAG_agentico.ipynb)

Este notebook es el pipeline completo, de punta a punta, y llama a los mismos módulos de
[`proyecto1/rag/`](https://github.com/JeanCarloLond/offside-londono-garces-ospina/tree/main/proyecto1/rag): no hay una copia del código aquí. La lectura de los
resultados está en el README y en [`docs/rag-m3.md`](https://github.com/JeanCarloLond/offside-londono-garces-ospina/blob/main/proyecto1/docs/rag-m3.md).

**Entorno:** Colab gratuito con GPU T4. Sin claves ni APIs de pago: todos los modelos son abiertos y
locales.

## 0 · Setup

Semilla fija (`SEED = 42`), decodificación greedy y versiones exactas de librerías
(`requirements-m3.txt`) y de modelos (commit de Hugging Face en `rag/config.py`).

`REPRODUCIR = True` vuelve a correr **todos** los sistemas sobre el eval set y a evaluarlos (harness +
RAGAS): en una T4 son del orden de una hora. Con `False` el notebook usa las respuestas y tablas
guardadas en `rag/resultados/`, que son exactamente las que produjo esa corrida, y aun así reconstruye
el índice y ejecuta en vivo las demostraciones.

Las salidas guardadas en este notebook son de una ejecución en una GTX 1650 Ti (4 GB) desde el repo
local: las latencias de Colab serán otras, y las respuestas en vivo deben coincidir con las de
`rag/resultados/respuestas/` (la decodificación es greedy).

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPRODUCIR = False

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not Path("offside").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/JeanCarloLond/offside-londono-garces-ospina.git",
                "offside",
            ],
            check=True,
        )
    RAIZ = Path("offside").resolve()
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(RAIZ / "requirements-m3.txt")],
        check=True,
    )
else:  # ejecutado desde proyecto1/notebooks/ en un clon local
    RAIZ = Path.cwd().resolve().parents[1]

os.chdir(RAIZ / "proyecto1" / "rag")
sys.path.insert(0, os.getcwd())
sys.path.insert(0, str(RAIZ / "proyecto1" / "eval"))
print("cwd:", Path.cwd().relative_to(RAIZ))

cwd: proyecto1\rag


In [2]:
import json
import random

import config
import numpy as np
import pandas as pd
import torch

random.seed(config.SEED)
np.random.seed(config.SEED)
torch.manual_seed(config.SEED)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("generador:", config.GENERADOR, "| embeddings:", config.EMBEDDINGS)
print("reranker:", config.RERANKER, "| top-k:", config.TOP_K)

device: cuda
generador: Qwen/Qwen2.5-1.5B-Instruct | embeddings: intfloat/multilingual-e5-small
reranker: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1 | top-k: 4


## 1 · El corpus

Titular + entradilla de los feeds RSS de **Marca** (seis secciones) y **El Tiempo** (Colombia),
recolectados por nuestro propio script (`data_collection/collect_rss.py`) en dos cortes: agosto de 2026
(pretemporada e inicio de liga) y el **14 de octubre de 2026** (liga en marcha). Solo guardamos lo que
el medio publica para sindicación —nunca el artículo— con su enlace de vuelta: el texto sigue siendo
de cada medio (ver `docs/dataset.md#idioma-y-licencia`).

In [3]:
corpus = pd.read_json(config.CORPUS, lines=True)
largos = corpus["texto"].str.len()
print(f"documentos: {len(corpus)}  (= chunks: un fragmento es un chunk)")
print(f"fechas: {corpus['fecha'].min()} .. {corpus['fecha'].max()}")
print(
    f"caracteres por documento: mediana {largos.median():.0f}, p90 {largos.quantile(0.9):.0f}, "
    f"máximo {largos.max()}"
)
print()
print(corpus["fuente"].value_counts().to_string())

documentos: 748  (= chunks: un fragmento es un chunk)
fechas: 2026-05-09 .. 2026-10-14
caracteres por documento: mediana 162, p90 225, máximo 564

fuente
marca                         206
marca_premier                 140
marca_bundesliga              129
marca_segunda                 107
marca_laliga                   98
marca_champions                32
eltiempo_deportes              20
eltiempo_futbol_colombiano     16


## 2 · El pipeline de indexación: ingest → chunk → embed → store

- **ingest** (`ingest.py`): limpia el HTML del feed, quita el «Leer» final, deduplica por id y por
  texto y guarda la sección de la URL (en Marca es el club). El crudo no se versiona; `corpus.jsonl`
  sí, por eso en Colab se parte de ahí.
- **chunk**: un fragmento = un chunk, sin solapamiento (justificación abajo).
- **embed**: `intfloat/multilingual-e5-small`, con los prefijos `query:`/`passage:` de E5.
- **store**: Chroma persistente, similitud coseno. Además, un índice BM25 en memoria para la
  búsqueda híbrida.

La celda reconstruye el índice desde cero.

In [4]:
import time

from indice import Recuperador

t0 = time.perf_counter()
rec = Recuperador(forzar=True)
print(
    f"índice '{rec.coleccion.name}': {rec.coleccion.count()} chunks en "
    f"{time.perf_counter() - t0:.1f} s (incluye cargar embeddings y reranker)"
)
ej = rec.por_id[rec.dense("¿Vinicius puede jugar el Clásico?", 1)[0]]
print("\nlo que ve el generador:", ej.para_contexto)

índice 'offside_multilingual_e5_small': 748 chunks en 24.9 s (incluye cargar embeddings y reranker)

lo que ve el generador: (2026-10-14 · Marca · real madrid) Vinicius se pierde el Clásico: cuatro partidos de sanción. El Comité de Disciplina le impone una dura sanción por su roja ante el Villarreal


## 3 · Las tres decisiones, con argumentos de nuestro corpus

**Chunks: un fragmento RSS por chunk, sin solapamiento.** Los fragmentos ya son la unidad mínima con
sentido (mediana 162 caracteres, máximo 564) y en ellos el sujeto y el hecho suelen quedar en frases
distintas: «*Chaira, baja sensible para el Oviedo. El marroquí se perderá entre dos y tres meses*».
Partirlos separaría al jugador de su lesión; y como un chunk nunca corta una frase, no hay nada que
solapar.

**Embeddings: `multilingual-e5-small`.** Lo comparamos con dos alternativas sobre nuestras propias
preguntas (tabla de abajo). El modelo de paráfrasis se hunde porque nuestras preguntas y los
titulares no son paráfrasis: «¿Cuenta el Sevilla con Guridi?» frente a «El Sevilla prepara la visita
al Bernabéu sin Guridi». e5-base gana en recall@1 y MRR, justo lo que el reranker iguala, con más del
doble de parámetros.

**top-k = 4.** Cada pregunta se responde con una o dos noticias; con k = 4 el documento correcto
llega en 22 de 23 preguntas (y k = 10 no rescata la que falta). Más chunks no traen información y sí
distractores, que es justo lo que confunde a un generador de 1.5B.

In [5]:
print(pd.read_csv("resultados/embeddings_m3.csv").to_string(index=False))
print()
print(pd.read_csv("resultados/recuperacion_m3.csv").to_string(index=False))

                                                     modelo  recall@1  recall@2  recall@4  recall@10    mrr  no_llega_en_50
                             intfloat/multilingual-e5-small    0.8261    0.9565    0.9565     0.9565 0.8953               0
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2    0.4783    0.4783    0.6087     0.6522 0.5259               4
                              intfloat/multilingual-e5-base    0.9130    0.9565    0.9565     0.9565 0.9387               0

        recuperador  recall@1  recall@2  recall@4  recall@10    mrr  no_llega_en_50
              dense    0.8261    0.9565    0.9565     0.9565 0.8953               0
               bm25    0.9130    0.9130    0.9565     0.9565 0.9294               0
            hibrido    0.8696    0.9565    0.9565     0.9565 0.9158               0
       dense+rerank    0.9565    0.9565    0.9565     1.0000 0.9638               0
     hibrido+rerank    0.9565    0.9565    0.9565     1.0000 0.9638               0

## 4 · El eval set

27 preguntas escritas por nosotros sobre el corpus, cada una con `input`, `esperado` (la misma señal
estructurada que mide el harness de M2: categoría, impacto, equipo) y `criterio`, más la `referencia`
y sus afirmaciones para RAGAS. **4 no tienen respuesta en el corpus**: ahí lo esperado es la válvula.
3 necesitan aritmética de fechas. 17 son adversariales (conflictos temporales, pares mínimos,
negaciones, nombres parciales).

Aparte hay un **dev set** de 14 preguntas sobre otras noticias, que es donde se ajustaron los prompts:
nunca se ajustó nada mirando el eval set.

In [6]:
eval_set = json.load(open(config.EVAL_SET, encoding="utf-8"))
tabla = pd.DataFrame(
    [
        {
            "id": e["eval_id"],
            "pregunta": e["input"]["text"],
            "esperado": e["esperado"]["category"],
            "en_corpus": e["en_corpus"],
            "herramienta": e["necesita_herramienta"],
            "dificultad": e["tipo_dificultad"],
        }
        for e in eval_set
    ]
)
print(
    f"{len(eval_set)} casos | fuera del corpus: {(~tabla.en_corpus).sum()} | "
    f"adversariales: {sum(e['adversarial'] for e in eval_set)}"
)
tabla

27 casos | fuera del corpus: 4 | adversariales: 17


,id,pregunta,esperado,en_corpus,herramienta,dificultad
0,M3-01,¿Vinicius puede jugar el Clásico?,sancion_suspension,True,False,conflicto temporal
1,M3-02,¿Juega Vinicius contra la Roma en la Champions?,cambio_tactico,True,False,"par mínimo: misma persona, otra competición"
2,M3-03,¿Isaac Romero fue expulsado? ¿Con qué equipo juega?,sancion_suspension,True,False,normal
3,M3-04,¿A quién le falta el Alavés en la banda derecha?,sancion_suspension,True,False,el club solo está en la URL
4,M3-05,¿Quién fue expulsado en el Tottenham contra el Manchester United?,sancion_suspension,True,False,normal
5,M3-06,¿Cuenta el Sevilla con Guridi para jugar en el Bernabéu?,baja_confirmada,True,False,normal
6,M3-07,¿Nico Williams estará con el Athletic en Mestalla?,baja_confirmada,True,False,nombre parcial
7,M3-08,¿Juega Marco Asensio con el Fenerbahçe contra el Aston Villa?,baja_confirmada,True,False,normal
8,M3-09,¿Volverá Uche a jugar con el Getafe esta temporada?,baja_confirmada,True,False,normal; tentación de usar la herramienta sin necesidad
9,M3-10,¿Qué problema tiene el Córdoba en el extremo izquierdo?,baja_confirmada,True,False,pregunta sin nombre propio


## 5 · El RAG ingenuo: `sistema_rag(pregunta)`

Recupera el top-4 por similitud densa, arma el prompt aumentado y genera. El prompt base (`generador.py`,
versión v3) es **el mismo para todas las versiones** del sistema. Pide un JSON con un primer campo
`noticia`: el número de la noticia que responde, o 0 si ninguna. Con `noticia = 0` la respuesta es la
válvula, aplicada de forma determinista al parsear.

In [7]:
import generador as gen
import sistemas

g = gen.Generador()
sistemas._CACHE.update({"rec": rec, "gen": g})

for q in ["¿Vinicius puede jugar el Clásico?", "¿Kylian Mbappé se pierde el Clásico?"]:
    r = sistemas.sistema_rag(q)
    print(q)
    print(f"  -> {r['category']} | {r['impact']} | {r['team']}  ({r['latencia_s']:.1f} s)")
    print("    ", r["respuesta"])

¿Vinicius puede jugar el Clásico?
  -> baja_confirmada | negativo_bajo | Real Madrid  (13.3 s)
     El Comité de Disciplina le ha impuesto una sanción de cuatro partidos debido a su roja ante el Villarreal.


¿Kylian Mbappé se pierde el Clásico?
  -> baja_confirmada | negativo_bajo | Real Madrid  (11.7 s)
     Vinicius se pierde el Clásico: cuatro partidos de sanción.


### Las versiones del prompt base (ajustadas sobre el dev set)

| Versión | Qué cambia | Categoría correcta | Calla teniendo respuesta | Calla sin respuesta |
|---|---|---:|---:|---:|
| v1 | La válvula como una regla del prompt | 8/14 | 5/11 | 3/3 |
| v2 | + dos ejemplos resueltos (imita el ejemplo) | 3/14 | 8/11 | 3/3 |
| **v3** | Campo `noticia` primero, sin ejemplos | **9/14** | **0/11** | **0/3: nunca calla** |
| v4 | + «comprueba que la noticia nombra al jugador» | 8/14 | 3/11 | 0/3 |

Con un generador de 1.5B la válvula por prompt es **un dial, no un interruptor**. Congelamos v3 (la que
mejor contesta) y la válvula de las preguntas sin respuesta se resolvió aparte (sección 6).

In [8]:
print(pd.read_csv("resultados/ajuste_prompt_m3.csv").to_string(index=False))

prompt aciertos_categoria valvula_falsa valvula_correcta
    v1               8/14          5/11              3/3
    v2               3/14          8/11              3/3
    v3               9/14          0/11              0/3
    v4               8/14          3/11              0/3


## 6 · Diagnóstico del ingenuo → técnicas

La lista de fallos del RAG ingenuo sobre el eval set, separados con la pregunta del enunciado: ¿no
llegó el chunk, llegó y el modelo lo ignoró, o el dato no está en el corpus?

In [9]:
ing = {
    r["eval_id"]: r
    for r in map(json.loads, open("resultados/respuestas/rag_ingenuo.jsonl", encoding="utf-8"))
}
filas = []
for e in eval_set:
    r = ing[e["eval_id"]]
    rel = set(e["docs_relevantes"])
    pos = [i + 1 for i, d in enumerate(r["ids"]) if d in rel]
    filas.append(
        {
            "id": e["eval_id"],
            "esperado": e["esperado"]["category"],
            "ingenuo": r["category"],
            "ok": r["category"] == e["esperado"]["category"],
            "chunk relevante en": pos or ("no aplica" if not e["en_corpus"] else "NO LLEGÓ"),
            "respuesta": r["respuesta"][:80],
        }
    )
diag = pd.DataFrame(filas)
print(f"categoría correcta: {diag.ok.sum()}/{len(diag)}")
diag

categoría correcta: 8/27


,id,esperado,ingenuo,ok,chunk relevante en,respuesta
0,M3-01,sancion_suspension,baja_confirmada,False,"[1, 4]",El Comité de Disciplina le ha impuesto una sanción de cuatro partidos debido a s
1,M3-02,cambio_tactico,regreso_alta,False,"[1, 2]","Vinicius volverá al once de Mourinho en el Olímpico, pero se perderá cuatro encu"
2,M3-03,sancion_suspension,baja_confirmada,False,"[1, 4]","[1] Isaac Romero marcó el 2-3, mando callar a la grada y terminó expulsado. El d"
3,M3-04,sancion_suspension,baja_confirmada,False,[1],[1] Quique abre el casting para la banda derecha. El técnico deberá buscar un re
4,M3-05,sancion_suspension,sancion_suspension,True,[1],El Tottenham fue expulsado a los 3 minutos.
5,M3-06,baja_confirmada,baja_confirmada,True,[1],[1] El Sevilla prepara la visita al Bernabéu sin Guridi. El centrocampista sufre
6,M3-07,baja_confirmada,duda_fisica,False,"[1, 2]",Nico Williams no entrenará en Mestalla debido a una lesión.
7,M3-08,baja_confirmada,baja_confirmada,True,[1],[Marco Asensio] baja de última hora para enfrentarse al Aston Villa.
8,M3-09,baja_confirmada,baja_confirmada,True,[1],[1] El Getafe comunica ‘una grave lesión’ en la rodilla derecha de Uche. El club
9,M3-10,baja_confirmada,baja_confirmada,True,[1],El increíble agujero negro del extremo izquierdo del Córdoba está causado por la


| Falla del ingenuo | Casos | Técnica que la ataca |
|---|---|---|
| El chunk llega, pero un distractor (noticia vieja o parecida) va delante | Musiala, Celta, Ochoa, Bentancur | **Reranking con cross-encoder** |
| Nombres propios raros y términos exactos | dense pone ruido primero en Celta y Ochoa | **Búsqueda híbrida BM25 + dense (RRF)** |
| Inventa en las preguntas sin respuesta (válvula 0/4) | Mbappé, Lamine, Balón de Oro, cuota | **Válvula verificada** |
| Hay que calcular una fecha | Ochoa, Chaira | **Herramienta `calcular_fecha`** |
| El chunk no llega (titular metafórico) | Svilar / «El rapero con guantes» | HyDE y multi-query, **probadas y descartadas** |

HyDE y multi-query no rescatan M3-11 y empeoran el resto (MRR 0,90 → 0,74): el documento hipotético
de un 1.5B mete ruido. Están en `sistemas.py` para poder medirlas, pero no en el avanzado.

## 7 · El RAG avanzado: `sistema_rag_avanzado(pregunta)`

Misma firma, mismo generador, mismo prompt. Cambia la recuperación:

1. **Híbrida:** dense (top-20) + BM25 (top-20), fusionadas con Reciprocal Rank Fusion (k = 60).
2. **Reranking:** `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` reordena el pool y deja el top-4.
3. **Válvula verificada:** el mismo Qwen, usado como clasificador sí/no por logits (como el juez de
   M2), decide chunk a chunk si la noticia es la fuente para responder. Si no queda ninguno, el sistema
   calla **sin llamar al generador**. En el dev set acierta 13 de 14 con el umbral natural de 0,5.

Probamos antes usar el puntaje del cross-encoder como umbral de válvula: no separa
(`resultados/valvula_dev_m3.csv`), porque puntúa bajo las preguntas de disponibilidad que exigen
inferir («¿Puede jugar Sergio Álvarez?» frente a «rotura muscular»).

In [10]:
for q in [
    "¿Cómo está Musiala? ¿Puede volver a jugar con el Bayern?",
    "¿Kylian Mbappé se pierde el Clásico?",
]:
    r = sistemas.sistema_rag_avanzado(q)
    print(q)
    print("  verificador P(sí) por chunk:", r["traza"].get("verificador"))
    print(f"  -> {r['category']} | {r['impact']}  ({r['latencia_s']:.1f} s)")
    print("    ", r["respuesta"])

¿Cómo está Musiala? ¿Puede volver a jugar con el Bayern?
  verificador P(sí) por chunk: [0.997, 0.994, 0.999, 0.988]
  -> regreso_alta | neutro  (26.0 s)
     Kompany da detalles sobre el estado de salud de Musiala: "Las cosas van mejorando". El técnico del Bayern Múnich asegura que el jugador no se ha vuelto a desmayar. 'Hay que dar por hecho que, si alguna vez ocurre, también estará bajo control', dice.


¿Kylian Mbappé se pierde el Clásico?
  verificador P(sí) por chunk: [0.326, 0.238, 0.007, 0.147]
  -> irrelevante | neutro  (8.6 s)
     No encuentro esa información en las noticias recolectadas.


## 8 · La capa agéntica: `sistema_final(pregunta)`

**La herramienta:** `calcular_fecha(fecha_inicio, cantidad, unidad, fecha_objetivo)`. Las bajas llegan
como duraciones relativas a la fecha de la noticia («alrededor de tres meses», 27 de agosto) y el
apostador pregunta por un día concreto. Esa aritmética un 1.5B la hace mal; una función pura, nunca.

**El esquema** (formato de *function calling*) va en el prompt; el modelo propone el JSON y lo
parseamos con tolerancia (comillas simples, ```json, argumentos como string, fechas en español,
«tres», rangos «2-3»…). Si los argumentos no son válidos, el error vuelve al modelo; **tope de 3
pasos**.

**Cuándo se usa, y por qué así.** Intentamos que el LLM decidiera solo: con la plantilla nativa de
tools de Qwen dejó de contestar; con la herramienta en el prompt no la llamó nunca; con un paso de
planificación aparte, 0 de 4. Este modelo no sabe decidir **cuándo**. Por eso el cuándo es una regla
verificable (la pregunta nombra una fecha y un chunk trae una duración con número) y el **qué** —los
argumentos— lo propone el LLM. La respuesta final sale del mismo prompt base, con el resultado de la
herramienta añadido a las noticias.

In [11]:
import herramientas as herr

print(json.dumps(herr.ESQUEMA_CALCULAR_FECHA, ensure_ascii=False, indent=2))
print()
malformado = (
    "{'herramienta': 'calcular_fechas', 'argumentos': "
    '\'{"fecha_inicio": "27 de agosto de 2026", "cantidad": "tres", '
    '"unidad": "Meses", "fecha_objetivo": "1 de noviembre"}\'}'
)
llamada = herr.extraer_llamada(malformado)
print("JSON malformado del modelo ->", llamada)
print("resultado ->", herr.ejecutar(llamada))

{
  "type": "function",
  "function": {
    "name": "calcular_fecha",
    "description": "Suma un plazo (dias, semanas o meses) a una fecha y lo compara con una fecha objetivo. Usala SOLO si la pregunta pide saber si un jugador estara disponible en una fecha concreta Y la noticia da la duracion de su baja. Si la noticia ya responde la pregunta sin calcular nada, NO la uses.",
    "parameters": {
      "type": "object",
      "properties": {
        "fecha_inicio": {
          "type": "string",
          "description": "Fecha de publicacion de la noticia de la baja, AAAA-MM-DD."
        },
        "cantidad": {
          "type": "number",
          "description": "Duracion de la baja. Si es un rango (2-3), el valor maximo."
        },
        "unidad": {
          "type": "string",
          "enum": [
            "dias",
            "semanas",
            "meses"
          ]
        },
        "fecha_objetivo": {
          "type": "string",
          "description": "Fecha por la que pre

Prueba aislada del planificador con noticias **ficticias** (en el corpus real solo hay dos bajas con
duración numérica y las dos están en el eval set). Compara dejar que el LLM decida (`llm`) con la regla
(`regla`).

In [12]:
print(
    pd.read_csv("resultados/prueba_planificador_m3.csv")[
        [
            "modo",
            "pregunta",
            "necesita",
            "llama",
            "llamada_valida",
            "pasos",
            "lectura",
            "usa_resultado_sin_fecha",
            "usa_resultado_con_fecha",
        ]
    ].to_string(index=False)
)

 modo                                                      pregunta  necesita  llama  llamada_valida  pasos                                                                                                                                                                                                                                                lectura usa_resultado_sin_fecha usa_resultado_con_fecha
  llm                          ¿Podrá jugar Lunar el 20 de octubre?      True  False           False      1                                                                                                                                                                                                                                                     []                     NaN                     NaN
  llm ¿Cuenta el Atlético Ribera con Ferro para el 1 de septiembre?      True  False           False      1                                                                                       

In [13]:
for q in [
    "¿Puede contar el Oviedo con Chaira para el 29 de noviembre?",
    "¿Estará Aaron Ochoa disponible para un partido el 1 de noviembre?",
    "¿Volverá Uche a jugar con el Getafe esta temporada?",
]:
    r = sistemas.sistema_final(q)
    print(q)
    print(
        "  herramienta:",
        [(h["arguments"], h["resultado"].get("lectura", h["resultado"])) for h in r["herramientas"]]
        or "no la llama",
    )
    print(f"  -> {r['category']} | {r['impact']}  ({r['latencia_s']:.1f} s)")
    print("    ", r["respuesta"])

¿Puede contar el Oviedo con Chaira para el 29 de noviembre?
  herramienta: [({'fecha_inicio': '2026-08-08', 'cantidad': 'entre dos y tres meses', 'unidad': 'meses', 'fecha_objetivo': '2026-11-29'}, 'El 2026-11-29 es 21 dias DESPUES del fin del plazo (2026-11-08): para esa fecha el plazo de baja YA se habra cumplido, aunque no hay alta confirmada.')]
  -> baja_confirmada | negativo_bajo  (30.6 s)
     El marroquí se perderá entre dos y tres meses de competición.


¿Estará Aaron Ochoa disponible para un partido el 1 de noviembre?
  herramienta: no la llama
  -> irrelevante | neutro  (8.3 s)
     No encuentro esa información en las noticias recolectadas.


¿Volverá Uche a jugar con el Getafe esta temporada?
  herramienta: no la llama
  -> baja_confirmada | negativo_alto  (19.9 s)
     Uche no volverá a jugar con el Getafe esta temporada debido a una grave lesión en la rodilla derecha.


## 9 · Evaluación: el harness de M2 y RAGAS

**Fase 1** (`correr_sistemas.py`): cada sistema responde las 27 preguntas y se graba todo —señal,
respuesta, chunks, llamadas y latencia por consulta—. **Fase 2** (`evaluar_m3.py`): la función
`harness(eval_set, sistema)` de M2, sin tocar, con su juez (Qwen2.5-1.5B, rúbrica v1), y las cuatro
métricas de RAGAS. Son dos fases porque generador y juez no caben a la vez en una GPU pequeña, y así el
juez no entra en la latencia.

In [14]:
del g
sistemas._CACHE.clear()
torch.cuda.empty_cache()

if REPRODUCIR:
    for paso in [
        ["diagnostico_recuperacion.py", "--con-llm"],
        ["ajuste_prompt.py"],
        ["calibrar_umbral.py"],
        ["prueba_planificador.py"],
        ["correr_sistemas.py"],
        ["evaluar_m3.py"],
    ]:
        subprocess.run([sys.executable, *paso], check=True)
else:
    print("Usando los resultados guardados en rag/resultados/ (REPRODUCIR = False).")

Usando los resultados guardados en rag/resultados/ (REPRODUCIR = False).


### La tabla evolutiva

In [15]:
sc = pd.read_csv("resultados/scorecard_m3.csv")
cols = [
    "sistema",
    "d1_f1_macro",
    "d2_juez_media",
    "d3_tasa_accionable",
    "latencia_mediana_s",
    "valvula_correcta",
    "valvula_falsa",
    "herramienta_cuando_hace_falta",
    "herramienta_sin_hacer_falta",
]
principal = ["m2_lexico", "m2_lora", "llm_sin_rag", "rag_ingenuo", "rag_avanzado", "sistema_final"]
sc.set_index("sistema").loc[principal, cols[1:]]

,d1_f1_macro,d2_juez_media,d3_tasa_accionable,latencia_mediana_s,valvula_correcta,valvula_falsa,herramienta_cuando_hace_falta,herramienta_sin_hacer_falta
sistema,,,,,,,,
m2_lexico,0.1101,2.519,0.1852,0.000,0/4,0/23,0/3,0
m2_lora,0.0601,2.519,0.1481,0.062,0/4,0/23,0/3,0
llm_sin_rag,0.0259,2.593,0.1111,9.267,3/4,19/23,0/3,0
rag_ingenuo,0.1389,2.593,0.3704,15.128,0/4,0/23,0/3,0
rag_avanzado,0.2890,2.926,0.5185,22.074,3/4,4/23,0/3,0
sistema_final,0.2890,2.926,0.5185,22.225,3/4,4/23,1/3,0


### Ablación: una técnica a la vez

In [16]:
abl = [
    "rag_ingenuo",
    "rag_hibrido",
    "rag_rerank",
    "rag_verificado",
    "rag_hyde",
    "rag_avanzado_sin_hibrido",
    "rag_avanzado_sin_rerank",
    "rag_avanzado_sin_verificador",
    "rag_avanzado",
    "sistema_final",
]
sc.set_index("sistema").loc[[a for a in abl if a in set(sc.sistema)], cols[1:]]

,d1_f1_macro,d2_juez_media,d3_tasa_accionable,latencia_mediana_s,valvula_correcta,valvula_falsa,herramienta_cuando_hace_falta,herramienta_sin_hacer_falta
sistema,,,,,,,,
rag_ingenuo,0.1389,2.593,0.3704,15.128,0/4,0/23,0/3,0
rag_hibrido,0.2662,2.667,0.4074,15.461,0/4,0/23,0/3,0
rag_rerank,0.2548,2.704,0.5185,16.322,1/4,0/23,0/3,0
rag_verificado,0.2655,2.889,0.4815,21.623,3/4,3/23,0/3,0
rag_hyde,0.1850,2.481,0.3333,26.132,0/4,0/23,0/3,0
rag_avanzado_sin_hibrido,0.2944,2.926,0.5185,22.646,3/4,4/23,0/3,0
rag_avanzado_sin_rerank,0.2259,2.926,0.4815,18.707,3/4,3/23,0/3,0
rag_avanzado_sin_verificador,0.1940,2.630,0.4815,13.546,0/4,0/23,0/3,0
rag_avanzado,0.2890,2.926,0.5185,22.074,3/4,4/23,0/3,0


### RAGAS

Implementación propia de las cuatro métricas (`ragas_local.py`) con las definiciones de RAGAS y juez
**Qwen2.5-1.5B-Instruct** leyendo logits sí/no. Sobre las 23 preguntas con respuesta en el corpus;
las 4 sin respuesta se leen con la válvula. `context_precision_ids` es la misma métrica calculada
con los documentos relevantes anotados a mano: sirve para comprobar al juez.

In [17]:
print(pd.read_csv("resultados/ragas_m3.csv").to_string(index=False))
det = json.load(open("resultados/ragas_m3_detalle.json", encoding="utf-8"))
s = det["sanity_check"]
print(f"\nSanity check del juez RAGAS: {s['aciertos']}/{s['total']}")
for c in s["casos"]:
    print(
        f"  {'OK ' if c['acierta'] else 'XX '}{c['tipo']:<22} p(sí)={c['p_si']:.2f}  {c['afirmacion']}"
    )

      sistema  n_casos  faithfulness  n_faithfulness  answer_relevancy  context_precision  context_recall  context_precision_ids                       juez
  rag_ingenuo       23        0.6812              23            0.8547             0.8152          0.8841                 0.8696 Qwen/Qwen2.5-1.5B-Instruct
 rag_avanzado       23        0.7746              19            0.7118             0.8261          0.7428                 0.8261 Qwen/Qwen2.5-1.5B-Instruct
sistema_final       23        0.8272              19            0.7114             0.8261          0.7536                 0.8261 Qwen/Qwen2.5-1.5B-Instruct

Sanity check del juez RAGAS: 8/8
  OK directa                p(sí)=1.00  Lunar no jugará el derbi.
  OK directa                p(sí)=1.00  Lunar estará tres semanas de baja.
  OK signo invertido        p(sí)=0.17  Lunar jugará el derbi.
  OK categoría cambiada     p(sí)=0.01  Lunar está sancionado.
  OK número cambiado        p(sí)=0.01  Lunar estará dos meses de baja.
  O

## 10 · La lectura

Qué técnica movió qué, cuánto costó, qué falla queda y qué pasó con la válvula: en el
[README](https://github.com/JeanCarloLond/offside-londono-garces-ospina#m3--rag-que-recupera-actúa-y-se-evalúa) y en detalle en
[`docs/rag-m3.md`](https://github.com/JeanCarloLond/offside-londono-garces-ospina/blob/main/proyecto1/docs/rag-m3.md).

*SI4006 · Universidad EAFIT · Módulo 3 · Equipo Offside*